# Phase 2 – Réseau de neurones (MLPRegressor)

Notebook pédagogique : étapes 3 → 5 du pipeline machine learning sur les 20 arrondissements parisiens.

**Baseline (phase 1) :** KMeans K=3 → One-Hot Encoding → LinearRegression  
**Phase 2 (ce notebook) :** `X/y` → split train/test → `MLPRegressor` → évaluation + comparaison

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.modeling import (
    load_master_arrondissements,
    create_feature_response_arrays,
    create_train_test_datasets,
    build_neural_network_pipeline,
    evaluate_regression_predictions,
    run_phase2_neural_network_pipeline,
)

print('Setup OK')

## Étape 3 – Chargement des données et création de X et y

La table `master_arrondissements` contient les 20 arrondissements de Paris avec leurs features agrégées.

In [ ]:
master_df = load_master_arrondissements()
print(f'Dimensions : {master_df.shape}')
master_df.head()

In [ ]:
X, y = create_feature_response_arrays(master_df)

print(f'X.shape = {X.shape}  (attendu : (20, 7))')
print(f'y.shape = {y.shape}  (attendu : (20,))')
print(f'NaN dans X : {X.isna().any().any()}')
print(f'NaN dans y : {y.isna().any()}')
X.describe()

## Étape 4 – Split train/test

`train_test_split(test_size=0.2, random_state=42)` — sans stratification (régression).  
Avec 20 observations : **16 train / 4 test**.

In [ ]:
datasets = create_train_test_datasets(X, y)
X_train = datasets['X_train']
X_test  = datasets['X_test']
y_train = datasets['y_train']
y_test  = datasets['y_test']

print(f'Train : {len(X_train)} lignes  (attendu : 16)')
print(f'Test  : {len(X_test)} lignes   (attendu : 4)')

pd.DataFrame([
    {'split': 'train', 'n_rows': len(X_train), 'y_min': y_train.min(), 'y_max': y_train.max(), 'y_mean': y_train.mean()},
    {'split': 'test',  'n_rows': len(X_test),  'y_min': y_test.min(),  'y_max': y_test.max(),  'y_mean': y_test.mean()},
])

## Étape 5a – Entraînement du réseau de neurones

Architecture : `StandardScaler` → `MLPRegressor(hidden_layer_sizes=(8, 4), activation='relu', solver='adam', alpha=0.001, max_iter=5000)`

In [ ]:
pipeline = build_neural_network_pipeline()
pipeline.fit(X_train, y_train)

print('Pipeline entraîné')
print('Étapes :', [name for name, _ in pipeline.steps])

## Étape 5b – Prédictions

In [ ]:
import numpy as np

y_pred_train = pipeline.predict(X_train)
y_pred_test  = pipeline.predict(X_test)

train_results = pd.DataFrame({
    'arrondissement': master_df.loc[X_train.index, 'arrondissement_name'].values,
    'y_observed':    y_train.values,
    'y_predicted':   y_pred_train,
    'residual':      y_train.values - y_pred_train,
    'split':         'train',
})

test_results = pd.DataFrame({
    'arrondissement': master_df.loc[X_test.index, 'arrondissement_name'].values,
    'y_observed':    y_test.values,
    'y_predicted':   y_pred_test,
    'residual':      y_test.values - y_pred_test,
    'split':         'test',
})

all_results = pd.concat([train_results, test_results]).reset_index(drop=True)
all_results.round(1)

## Étape 5c – Métriques

On calcule R², RMSE et MAE sur le **train set**, le **test set**, et en **LOOCV** sur l'ensemble complet.

In [ ]:
from sklearn.model_selection import LeaveOneOut, cross_val_predict

train_metrics = {'split': 'train', **evaluate_regression_predictions(y_train, y_pred_train)}
test_metrics  = {'split': 'test',  **evaluate_regression_predictions(y_test,  y_pred_test)}

y_pred_loocv = cross_val_predict(build_neural_network_pipeline(), X, y, cv=LeaveOneOut())
loocv_metrics = {'split': 'loocv', **evaluate_regression_predictions(y, y_pred_loocv)}

metrics_df = pd.DataFrame([train_metrics, test_metrics, loocv_metrics])
metrics_df.round(4)

## Étape 5d – Visualisations d'évaluation

In [ ]:
# Actual vs Predicted
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(train_results['y_observed'], train_results['y_predicted'],
           label='Train', color='steelblue', zorder=3)
ax.scatter(test_results['y_observed'], test_results['y_predicted'],
           label='Test', color='coral', marker='D', s=80, zorder=4)
lims = [
    min(all_results['y_observed'].min(), all_results['y_predicted'].min()) * 0.9,
    max(all_results['y_observed'].max(), all_results['y_predicted'].max()) * 1.1,
]
ax.plot(lims, lims, 'r--', linewidth=1, label='Ajustement parfait')
ax.set_xlabel('Y observé (nombre de corbeilles)')
ax.set_ylabel('Y prédit')
ax.set_title('MLP : Observé vs Prédit')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Residuals vs Predicted
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(train_results['y_predicted'], train_results['residual'],
           label='Train', color='steelblue')
ax.scatter(test_results['y_predicted'], test_results['residual'],
           label='Test', color='coral', marker='D', s=80)
ax.axhline(0, color='red', linestyle='--', linewidth=1)
ax.set_xlabel('Y prédit')
ax.set_ylabel('Résidu (observé − prédit)')
ax.set_title('MLP : Résidus vs Prédit')
ax.legend()
plt.tight_layout()
plt.show()

## Comparaison avec la baseline linéaire

La régression linéaire (KMeans + LinearRegression) est la **phase 1 / baseline**.  
Elle sert de point de comparaison pédagogique, non de méthode principale.

In [ ]:
from config import TABLES_DIR

baseline_metrics = pd.read_csv(TABLES_DIR / 'linear_regression_baseline_metrics.csv')
nn_metrics = pd.read_csv(TABLES_DIR / 'neural_network_metrics.csv')

print('=== MLP (phase 2) ===')
print(nn_metrics.to_string(index=False))
print()
print('=== Baseline linéaire (phase 1) ===')
print(baseline_metrics.to_string(index=False))

## Limitations et conclusions pédagogiques

- **20 observations** sont insuffisantes pour un réseau de neurones : le sur-apprentissage est inévitable (R² train positif, R² test négatif).
- La **LOOCV** est préférée au test split simple pour évaluer la généralisation sur un si petit jeu de données.
- Ce pipeline illustre comment implémenter un MLP avec scikit-learn, structurer un train/test split, et comparer rigoureusement deux approches.
- Pour des résultats exploitables, il faudrait passer à la maille IRIS (~1000 observations) ou intégrer des données supplémentaires.